In [1]:
import json
import sys
from pathlib import Path
import importlib

import pandas as pd

BASE_DIR = Path.cwd().parent

if str(BASE_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(BASE_DIR)
    )

import src.extractor
import src.validator

importlib.reload(
    src.extractor
)

importlib.reload(
    src.validator
)

from src.extractor import MeetingActionExtractor
from src.validator import validate_items

DATA_PATH = (
    BASE_DIR /
    "data" /
    "corpus.json"
)

with open(
    DATA_PATH,
    "r",
    encoding="utf-8"
) as f:

    corpus = json.load(f)

print(
    "Corpus loaded"
)

print(
    "Meetings:",
    len(corpus["reunions"])
)

c:\Users\Anshu\anaconda3\envs\fraud_x\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Corpus loaded
Meetings: 8


In [2]:
meeting_records = []

for meeting_index, meeting in enumerate(
    corpus["reunions"]
):

    texts = []

    for reply in meeting["repliques"]:

        text = reply.get(
            "texte",
            ""
        )

        if text:
            texts.append(
                text
            )

    transcript = "\n".join(
        texts
    )

    meeting_records.append({
        "meeting_id": meeting_index + 1,
        "transcript": transcript
    })

meetings_df = pd.DataFrame(
    meeting_records
)

display(
    meetings_df[
        [
            "meeting_id",
            "transcript"
        ]
    ]
)

,meeting_id,transcript
0,1,"Right, let us get started, I have a hard stop ..."
1,2,"Everyone here? Good.\nUnderstood.\nElsa, can y..."
2,3,"Quick round on last week first.\nSorry, I miss..."
3,4,"Right, let us get started, I have a hard stop ..."
4,5,"Everyone here? Good.\nFootfall was flat, same ..."
5,6,Quick round on last week first.\nThat is fair....
6,7,"Right, let us get started, I have a hard stop ..."
7,8,Everyone here? Good.\nThat is fair.\nSomeone h...


In [3]:
for _, row in meetings_df.iterrows():

    print(
        f"Meeting {row['meeting_id']}: "
        f"{len(row['transcript'])} characters"
    )

Meeting 1: 809 characters
Meeting 2: 792 characters
Meeting 3: 638 characters
Meeting 4: 787 characters
Meeting 5: 781 characters
Meeting 6: 790 characters
Meeting 7: 815 characters
Meeting 8: 855 characters


In [4]:
extractor = MeetingActionExtractor()

print(
    "FLAN-T5 extractor loaded"
)

print(
    "Candidate method:",
    hasattr(
        extractor,
        "find_candidates"
    )
)

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 1108.44it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


FLAN-T5 extractor loaded
Candidate method: True


In [5]:
test_transcript = meetings_df.loc[
    meetings_df["meeting_id"] == 1,
    "transcript"
].iloc[0]

print(
    test_transcript
)

Right, let us get started, I have a hard stop at eleven.
Two people are off next week, so we will be short handed.
I will update the pricing on the kiosk by Wednesday.
I will take down the summer banner.
Nadia, can you call the sign shop about the awning?
Sure, that is mine.
Agreed, let us keep it short today.
And the mailbox at the old address still has to be emptied, that one is still open.
Should we put a neon sign in the window?
No, we already decided against it.
Footfall was flat, same as the week before.
I already reset the alarm codes last week, that one is closed.
Padma, please get the pergola quote moving.
Will do.
Actually, forget the pergola job, the landlord said no.
Someone has to see to this: the gate keypad code has to be rotated.
Anything else before we close?
Fine, that is us then.


In [6]:
candidates = extractor.find_candidates(
    test_transcript
)

print(
    "Candidate sentences:",
    len(candidates)
)

for i, sentence in enumerate(
    candidates,
    start=1
):

    print(
        f"{i}. {sentence}"
    )

Candidate sentences: 8
1. Two people are off next week, so we will be short handed.
2. I will update the pricing on the kiosk by Wednesday.
3. I will take down the summer banner.
4. Nadia, can you call the sign shop about the awning?
5. And the mailbox at the old address still has to be emptied, that one is still open.
6. Padma, please get the pergola quote moving.
7. Will do.
8. Someone has to see to this: the gate keypad code has to be rotated.


In [7]:
test_actions = extractor.extract(
    test_transcript
)

print(
    "Extracted action items:",
    len(test_actions)
)

for i, item in enumerate(
    test_actions,
    start=1
):

    print(
        f"\nItem {i}"
    )

    print(
        item
    )

Extracted action items: 8

Item 1
{'task': 'Two people are off next week, so we will be short handed.', 'owner': None, 'deadline': None, 'status': 'pending', 'evidence': 'Two people are off next week, so we will be short handed.'}

Item 2
{'task': 'I will update the pricing on the kiosk by Wednesday.', 'owner': None, 'deadline': 'by Wednesday', 'status': 'pending', 'evidence': 'I will update the pricing on the kiosk by Wednesday.'}

Item 3
{'task': 'I will take down the summer banner.', 'owner': None, 'deadline': None, 'status': 'pending', 'evidence': 'I will take down the summer banner.'}

Item 4
{'task': 'Nadia, can you call the sign shop about the awning?', 'owner': 'Nadia', 'deadline': None, 'status': 'pending', 'evidence': 'Nadia, can you call the sign shop about the awning?'}

Item 5
{'task': 'And the mailbox at the old address still has to be emptied, that one is still open.', 'owner': None, 'deadline': None, 'status': 'pending', 'evidence': 'And the mailbox at the old address s

In [8]:
validated_actions = validate_items(
    test_actions
)

print(
    "Validated action items:",
    len(validated_actions)
)

if validated_actions:

    result_df = pd.DataFrame(
        validated_actions
    )

    display(
        result_df[
            [
                "task",
                "owner",
                "deadline",
                "status",
                "confidence",
                "evidence",
                "warnings"
            ]
        ]
    )

else:

    print(
        "No validated action items."
    )

Validated action items: 8


,task,owner,deadline,status,confidence,evidence,warnings
0,"Two people are off next week, so we will be sh...",NaN,NaN,pending,0.75,"Two people are off next week, so we will be sh...","[Missing owner, Missing deadline]"
1,I will update the pricing on the kiosk by Wedn...,NaN,by Wednesday,pending,0.90,I will update the pricing on the kiosk by Wedn...,[Missing owner]
2,I will take down the summer banner.,NaN,NaN,pending,0.75,I will take down the summer banner.,"[Missing owner, Missing deadline]"
3,"Nadia, can you call the sign shop about the aw...",Nadia,NaN,pending,0.90,"Nadia, can you call the sign shop about the aw...",[Missing deadline]
4,And the mailbox at the old address still has t...,NaN,NaN,pending,0.75,And the mailbox at the old address still has t...,"[Missing owner, Missing deadline]"
5,"Padma, please get the pergola quote moving.",Padma,NaN,pending,0.90,"Padma, please get the pergola quote moving.",[Missing deadline]
6,Will do.,NaN,NaN,pending,0.75,Will do.,"[Missing owner, Missing deadline]"
7,Someone has to see to this: the gate keypad co...,NaN,NaN,pending,0.75,Someone has to see to this: the gate keypad co...,"[Missing owner, Missing deadline]"


In [9]:
all_results = []

for _, row in meetings_df.iterrows():

    print(
        f"Processing Meeting {row['meeting_id']}..."
    )

    extracted = extractor.extract(
        row["transcript"]
    )

    validated = validate_items(
        extracted
    )

    for item in validated:

        all_results.append({
            "meeting_id": row["meeting_id"],
            "task": item["task"],
            "owner": item["owner"],
            "deadline": item["deadline"],
            "status": item["status"],
            "confidence": item["confidence"],
            "evidence": item["evidence"],
            "warnings": item["warnings"]
        })

print(
    "Total extracted action items:",
    len(all_results)
)

Processing Meeting 1...
Processing Meeting 2...
Processing Meeting 3...
Processing Meeting 4...
Processing Meeting 5...
Processing Meeting 6...
Processing Meeting 7...
Processing Meeting 8...
Total extracted action items: 54


In [10]:
columns = [
    "meeting_id",
    "task",
    "owner",
    "deadline",
    "status",
    "confidence",
    "evidence",
    "warnings"
]

actions_df = pd.DataFrame(
    all_results,
    columns=columns
)

display(
    actions_df
)

,meeting_id,task,owner,deadline,status,confidence,evidence,warnings
0,1,"Two people are off next week, so we will be sh...",NaN,NaN,pending,0.75,"Two people are off next week, so we will be sh...","[Missing owner, Missing deadline]"
1,1,I will update the pricing on the kiosk by Wedn...,NaN,by Wednesday,pending,0.90,I will update the pricing on the kiosk by Wedn...,[Missing owner]
2,1,I will take down the summer banner.,NaN,NaN,pending,0.75,I will take down the summer banner.,"[Missing owner, Missing deadline]"
3,1,"Nadia, can you call the sign shop about the aw...",Nadia,NaN,pending,0.90,"Nadia, can you call the sign shop about the aw...",[Missing deadline]
4,1,And the mailbox at the old address still has t...,NaN,NaN,pending,0.75,And the mailbox at the old address still has t...,"[Missing owner, Missing deadline]"
5,1,"Padma, please get the pergola quote moving.",Padma,NaN,pending,0.90,"Padma, please get the pergola quote moving.",[Missing deadline]
6,1,Will do.,NaN,NaN,pending,0.75,Will do.,"[Missing owner, Missing deadline]"
7,1,Someone has to see to this: the gate keypad co...,NaN,NaN,pending,0.75,Someone has to see to this: the gate keypad co...,"[Missing owner, Missing deadline]"
8,2,And the wash bay hose has to be replaced befor...,NaN,NaN,pending,0.75,And the wash bay hose has to be replaced befor...,"[Missing owner, Missing deadline]"
9,2,"Nora, please swap the vending supplier.",Nora,NaN,pending,0.90,"Nora, please swap the vending supplier.",[Missing deadline]


In [11]:
print(
    "Total action items:",
    len(actions_df)
)

print(
    "Meetings processed:",
    actions_df["meeting_id"].nunique()
    if not actions_df.empty
    else 0
)

print(
    "Items with owner:",
    actions_df["owner"].notna().sum()
    if not actions_df.empty
    else 0
)

print(
    "Items with deadline:",
    actions_df["deadline"].notna().sum()
    if not actions_df.empty
    else 0
)

print(
    "Average confidence:",
    round(
        actions_df["confidence"].mean(),
        3
    )
    if not actions_df.empty
    else 0
)

Total action items: 54
Meetings processed: 8
Items with owner: 20
Items with deadline: 8
Average confidence: 0.824


In [12]:
if actions_df.empty:

    print(
        "No action items available for owner analysis."
    )

else:

    missing_owner = actions_df[
        actions_df["owner"].isna()
    ]

    print(
        "Action items with missing owner:",
        len(missing_owner)
    )

    display(
        missing_owner[
            [
                "meeting_id",
                "task",
                "deadline",
                "confidence"
            ]
        ]
    )

Action items with missing owner: 34


,meeting_id,task,deadline,confidence
0,1,"Two people are off next week, so we will be sh...",NaN,0.75
1,1,I will update the pricing on the kiosk by Wedn...,by Wednesday,0.90
2,1,I will take down the summer banner.,NaN,0.75
4,1,And the mailbox at the old address still has t...,NaN,0.75
6,1,Will do.,NaN,0.75
7,1,Someone has to see to this: the gate keypad co...,NaN,0.75
8,2,And the wash bay hose has to be replaced befor...,NaN,0.75
10,2,Will do.,NaN,0.75
14,2,I will measure the mezzanine for the new racking.,NaN,0.75
15,3,I will order a new shredder by next week.,by next week,0.90


In [13]:
if actions_df.empty:

    print(
        "No action items available for deadline analysis."
    )

else:

    missing_deadline = actions_df[
        actions_df["deadline"].isna()
    ]

    print(
        "Action items with missing deadline:",
        len(missing_deadline)
    )

    display(
        missing_deadline[
            [
                "meeting_id",
                "task",
                "owner",
                "confidence"
            ]
        ]
    )

Action items with missing deadline: 46


,meeting_id,task,owner,confidence
0,1,"Two people are off next week, so we will be sh...",NaN,0.75
2,1,I will take down the summer banner.,NaN,0.75
3,1,"Nadia, can you call the sign shop about the aw...",Nadia,0.90
4,1,And the mailbox at the old address still has t...,NaN,0.75
5,1,"Padma, please get the pergola quote moving.",Padma,0.90
6,1,Will do.,NaN,0.75
7,1,Someone has to see to this: the gate keypad co...,NaN,0.75
8,2,And the wash bay hose has to be replaced befor...,NaN,0.75
9,2,"Nora, please swap the vending supplier.",Nora,0.90
10,2,Will do.,NaN,0.75


In [14]:
if actions_df.empty:

    print(
        "No confidence values available."
    )

else:

    print(
        actions_df["confidence"].describe()
    )

count    54.000000
mean      0.824074
std       0.087276
min       0.750000
25%       0.750000
50%       0.750000
75%       0.900000
max       1.000000
Name: confidence, dtype: float64


In [15]:
if actions_df.empty:

    print(
        "No low-confidence items available."
    )

else:

    low_confidence = actions_df[
        actions_df["confidence"] < 0.70
    ]

    print(
        "Low-confidence items:",
        len(low_confidence)
    )

    display(
        low_confidence[
            [
                "meeting_id",
                "task",
                "owner",
                "deadline",
                "confidence",
                "warnings"
            ]
        ]
    )

Low-confidence items: 0


,meeting_id,task,owner,deadline,confidence,warnings


In [16]:
REPORTS_DIR = (
    BASE_DIR /
    "reports"
)

REPORTS_DIR.mkdir(
    exist_ok=True
)

OUTPUT_PATH = (
    REPORTS_DIR /
    "extracted_action_items.csv"
)

actions_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(
    f"Saved to: {OUTPUT_PATH}"
)

Saved to: c:\Users\Anshu\Documents\Zeravia AI projects\AI Meeting\reports\extracted_action_items.csv


In [17]:
JSON_PATH = (
    REPORTS_DIR /
    "extracted_action_items.json"
)

with open(
    JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        all_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    f"Saved to: {JSON_PATH}"
)

Saved to: c:\Users\Anshu\Documents\Zeravia AI projects\AI Meeting\reports\extracted_action_items.json
